In [ ]:
###############################################
##                                           ##
##           IMPORTS & RANDOM SEED           ##
##                                           ##
###############################################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.utils import simple_preprocess
from gensim.parsing.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import gensim
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize

SEED = 16
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
#############################################
##                                         ##
##              DATA FETCHING              ##
##                                         ##
#############################################

# data_dir = "./data/"
data_dir = "/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/"
train_df = pd.read_csv(f"{data_dir}train_dataset.csv")
val_df = pd.read_csv(f"{data_dir}val_dataset.csv")
test_df = pd.read_csv(f"{data_dir}test_dataset.csv")

# debugging
train_df.head()

In [ ]:
#############################################
##                                         ##
##           DATA PRE-PROCESSING           ##
##                                         ##
#############################################

import re
import html
import contractions
from spellchecker import SpellChecker

# initialize spell checker
spell = SpellChecker(distance=1)

def remove_contractions(text):
    # expand contractions (e.g haven't -> have not)
    return contractions.fix(text)

def my_spellcheck(text):
    words = text.split()
    corrected_words = []
    for word in words:
        # if it's wrong
        if word not in spell:
            test_word = spell.correction(word)
            # found correction
            if test_word is not None:
                corrected_words.append(test_word)
            # didn't find correction
            else:
                corrected_words.append(word)
        # else, if it's correct, append it to the corrected
        else:
            corrected_words.append(word)
    return " ".join(corrected_words)

def text_cleanup(text):
    text = text.lower() # lowercase everything
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email', text) # handle email adresses
    text = re.sub(r'http[s]?://\S+', 'url', text) # handle urls
    text = re.sub(r'@\w+', 'username', text) # handle mentions
    text = re.sub(r'#(\w+)', r'\1', text) # handle hashtags: remove the symbol, keep the text
    text = html.unescape(text) # handling html tags
    text = re.sub(r'(\w)\1{2,}', r'\1\1', text) # handling unecessarily long words (e.g. soooooo -> soo)
    text = remove_contractions(text) # handling contractions (haven't -> have not)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # remove anything that is not english, a number or a whitespace
    text = my_spellcheck(text) # spellchecking using pyspellchecker
    return text

# apply cleanup to both datasets
train_df["Text"] = train_df["Text"].apply(text_cleanup)
val_df["Text"] = val_df["Text"].apply(text_cleanup)
test_df["Text"] = test_df["Text"].apply(text_cleanup)

# debugging
# print("Cleaned, pre-processed data:")
train_df.head()

In [ ]:
################################################
###                                          ###
###       TOKENIZATION & VECTORIZATION       ###
###                                          ###
################################################

# word_to_vector model for vectorizing tokens

import gensim.downloader as api
from gensim.models import KeyedVectors
# only need to do these once, then it'll then be saved locally
# word2vec_model = api.load("word2vec-google-news-300")
# word2vec_model.save("./word2vec_google_news.kv")
word2vec_model = KeyedVectors.load_word2vec_format("/kaggle/input/google-word2vec/GoogleNews-vectors-negative300.bin",binary=True)

def tokenize(df):
    df["Text"] = df["Text"].astype(str).apply(lambda x: x.split())
    return df

def vectorize(df):
    vectors = []
    for tokens in df["Text"]:
        word_vectors = []
        for token in tokens:
            if token in word2vec_model:
                word_vectors.append(word2vec_model[token])
        if word_vectors:
            # clculating the mean value of this document's vectors
            document_vector = np.mean(word_vectors, axis=0)
        else:
            # if no words in the current document appear in the dictionary,
            # a vector of zeros is chosen
            document_vector = np.zeros(word2vec_model.vector_size)
        vectors.append(document_vector)
    df["Vector"] = vectors
    return df



train_df = tokenize(train_df)
val_df = tokenize(val_df)
test_df = tokenize(test_df)

train_df = vectorize(train_df)
val_df = vectorize(val_df)
test_df = vectorize(test_df)

train_df.head()

In [ ]:
########################################################
###                                                  ###
###         STORING TRAINING DATA IN TENSORS         ###
###                                                  ###
########################################################

# vectors
x_df = pd.DataFrame(train_df, columns=[train_df.columns[-1]])
# labels
y_df = pd.DataFrame(train_df, columns=[train_df.columns[2]])

x_df['Vector'] = x_df['Vector'].apply(lambda v: np.array(v, dtype=np.float32))
x_train_numpy_array = np.stack(x_df['Vector'].values)

x = torch.tensor(x_train_numpy_array, dtype=torch.float32)
y = torch.tensor(y_df.values, dtype=torch.float32)

# debugging
# x_df.head()
# y_df.head()
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")

In [ ]:
########################################################
###                                                  ###
###        STORING VALIDATION DATA IN TENSORS        ###
###                                                  ###
########################################################

# vectors
x_val_df = pd.DataFrame(val_df, columns=[val_df.columns[-1]])
# labels
y_val_df = pd.DataFrame(val_df, columns=[val_df.columns[2]])

x_val_df['Vector'] = x_val_df['Vector'].apply(lambda v: np.array(v, dtype=np.float32))
x_val_numpy_array = np.stack(x_val_df['Vector'].values)

x_val = torch.tensor(x_val_numpy_array, dtype=torch.float32)
y_val = torch.tensor(y_val_df.values, dtype=torch.float32)

In [ ]:
# neural netwrok class definition

class Net(nn.Module):
    def __init__(self, D_in, H1, H2, H3, D_out, activation_name, dropout_rate):
        super(Net, self).__init__()

        activations = {
            "ReLU": nn.ReLU(),
            "LeakyReLU": nn.LeakyReLU(),
            "Tanh": nn.Tanh(),
            "Sigmoid": nn.Sigmoid()
        }
        act = activations[activation_name]

        self.model = nn.Sequential(
            nn.Linear(D_in, H1),
            act,
            nn.Dropout(dropout_rate),
            nn.Linear(H1, H2),
            act,
            nn.Dropout(dropout_rate),
            nn.Linear(H2, H3),
            act,
            nn.Dropout(dropout_rate),
            nn.Linear(H3, D_out),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
###############################################
###                                         ###
###         NEURAL NETWORK CREATION         ###
###                                         ###
###############################################

#Define layer sizes
D_in = x.shape[1] #size of the input sample
H1 = 128
H2 = 64
H3 = 32
D_out = 1

#Define Hyperparameters
learning_rate = 1e-4
activation_name = "ReLU"
dropout_rate = 0.2

my_model = Net(D_in, H1, H2, H3, D_out, activation_name, dropout_rate)
loss_func = nn.MSELoss(reduction='sum') # You can also try BCELoss and BCEWithLogitsLoss
optimizer = torch.optim.SGD(my_model.parameters(), lr=learning_rate) # You can also try Adam and AdamW

dataset = torch.utils.data.TensorDataset(x, y) #class to represent the data as list of tensors. x=input_features, y=labels
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
#########################################
###                                   ###
###          SAMPLE_TRAINING          ###
###                                   ###
#########################################

# debugging
# epochs_range = 5

# for epoch in range(epochs_range):
#     batch_losses = []

#     for x_batch, y_batch in dataloader:
#         y_pred = my_model(x_batch)

#         loss = loss_func(y_pred, y_batch)
#         batch_losses.append(loss.item())
#         # Delete previously stored gradients
#         optimizer.zero_grad()
#         # Perform backpropagation starting from the loss calculated in this epoch
#         loss.backward()
#         # Update model's weights based on the gradients calculated during backprop
#         optimizer.step()

#     print(f"Epoch {epoch:3}: Loss = {sum(batch_losses)/len(dataloader):.5f}")

In [ ]:
######################################
###                                ###
###             OPTUNA             ###
###                                ###
######################################

import optuna

def suggest_hyperparameters(trial):
    H1 = trial.suggest_int("H1", 64, 256, step=32)
    H2 = H1//2
    H3 = H1//4

    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW"])
    activation_name = trial.suggest_categorical("activation", ["ReLU", "LeakyReLU", "Tanh"])

    return H1, H2, H3, learning_rate, batch_size, dropout_rate, weight_decay, optimizer_name, activation_name


def objective(trial):
    H1, H2, H3, learning_rate, batch_size, dropout_rate, weight_decay, optimizer_name, activation_name = suggest_hyperparameters(trial)
    
    D_in = x.shape[1]
    D_out = 1

    model = Net(D_in, H1, H2, H3, D_out, activation_name, dropout_rate)

    loss_func = nn.BCELoss()

    # choosing optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    dataset = torch.utils.data.TensorDataset(x, y)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(5):
        batch_losses = []
        for x_batch, y_batch in dataloader:
            y_pred = model(x_batch)
            loss = loss_func(y_pred, y_batch)
            batch_losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_loss = sum(batch_losses)/len(dataloader)
        print(f"Epoch {epoch:3}: Loss = {epoch_loss:.5f}")

    return epoch_loss

# optuna_sample = optuna.create_study(direction = 'minimize' , study_name = 'lr-minim-sample')
# optuna_sample.optimize(objective, n_trials = 200) #the first parameter is the function that we want to optimise
# print('numbers of the finished trials:' , len(optuna_sample.trials))
# print('the best params:' , optuna_sample.best_trial.params)
# print('the best value:' , optuna_sample.best_value)

In [ ]:
# Using the output of optuna to perform the final training of the model
# H1 = optuna_sample.best_trial.params['H1']
# learning_rate = optuna_sample.best_trial.params['learning_rate']
# batch_size = optuna_sample.best_trial.params['batch_size']
# dropout_rate = optuna_sample.best_trial.params['dropout_rate']
# weight_decay = optuna_sample.best_trial.params['weight_decay']
# optimizer_name = optuna_sample.best_trial.params['optimizer']
# loss_func = nn.BCELoss()
# activation_name = optuna_sample.best_trial.params['activation']
# H2 = H1//2
# H3 = H1//4
# D_in = x.shape[1]
# D_out = 1

# the following are the best parameters that optuna produced over 200 trials
# hard coded for future runs
H1 = 256
H2 = H1//2
H3 = H1//4
learing_rate = 0.00045107211839375217
batch_size = 32
dropout_rate = 0.10020346423181177
weight_decay = 4.380515001378823e-06
optimizer_name = 'AdamW'
loss_func = nn.BCELoss()
activation_name = 'ReLU'
D_in = x.shape[1]
D_out = 1

my_model = Net(D_in, H1, H2, H3, D_out, activation_name, dropout_rate)

if optimizer_name == "Adam":
    optimizer = torch.optim.Adam(my_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
elif optimizer_name == "AdamW":
    optimizer = torch.optim.AdamW(my_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
elif optimizer_name == "SGD":
    optimizer = torch.optim.SGD(my_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

x_train = x
y_train = y
train_dataset = torch.utils.data.TensorDataset(x_train, y_train)
val_dataset = torch.utils.data.TensorDataset(x_val, y_val)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)

# used for learning curves
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

epochs_range = 5

for epoch in range(epochs_range):
    my_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        y_pred = my_model(x_batch)
        loss = loss_func(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predicted = (y_pred > 0.5).float()
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validation
    my_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x_val_batch, y_val_batch in val_loader:
            y_val_pred = my_model(x_val_batch)
            loss = loss_func(y_val_pred, y_val_batch)
            val_loss += loss.item()
            predicted = (y_val_pred > 0.5).float()
            val_correct += (predicted == y_val_batch).sum().item()
            val_total += y_val_batch.size(0)

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{epochs_range} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# evaluating accuracy, precision, f1 score etc
my_model.eval()
with torch.no_grad():
    val_output = my_model(x_val)
    prediction = (val_output >= 0.5).float()

accuracy = accuracy_score(y_val, prediction)
print(prediction.shape)
print(f"Accuracy: {accuracy}")
print("Classification Report:\n", classification_report(y_val, prediction))

In [ ]:
#######################################
####                               ####
####             PLOTS             ####
####                               ####
#######################################

from sklearn.metrics import confusion_matrix
import seaborn as sns

# learning curves

epochs = range(1, len(train_losses) + 1)
# Loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, 'bo-', label='Training Loss')
plt.plot(epochs, val_losses, 'ro-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accuracies, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_accuracies, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=120)
plt.show()

# comparison with previous project

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
tfidf_scores = [80.59, 81.00, 80.00, 80.0]  # Project 1
nn_scores = [76.87, 76.00, 79.00, 77.00]    # Project 2

x = range(len(metrics))
width = 0.35
fig, ax = plt.subplots()
ax.bar(x, tfidf_scores, width, label='TF-IDF + Logistic Regression', color='green')
ax.bar([i + width for i in x], nn_scores, width, label='Word2Vec + Neural Net', color='yellow')
ax.set_ylabel('Score (%)')
ax.set_title('Models comaprison')
ax.set_xticks([i + width/2 for i in x])
ax.set_xticklabels(metrics)
ax.legend()
plt.savefig('comparison.png', bbox_inches='tight', dpi=120)
plt.show()

# confusion matrix

cm = confusion_matrix(y_val, prediction)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="OrRd", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
#####################################
####                             ####
####      KAGGLE SUBMISSION      ####
####                             ####
#####################################

# vectors
x_test_df = pd.DataFrame(test_df, columns=[test_df.columns[-1]])

x_test_df['Vector'] = x_test_df['Vector'].apply(lambda v: np.array(v, dtype=np.float32))
x_test_numpy_array = np.stack(x_test_df['Vector'].values)

x_test = torch.tensor(x_test_numpy_array, dtype=torch.float32)

my_model.eval()
with torch.no_grad():
    y_test = my_model(x_test)
    test_prediction = (y_test >= 0.5).int()

submission_df = pd.DataFrame({
    "ID": test_df["ID"],
    "Label": test_prediction.view(-1).numpy()
})

submission_df.to_csv("./submission.csv", index=False)
print("Created submission csv file")